In [1]:
import torch
import dgl
import yaml

print("torch:", torch.__version__)
print("dgl:", dgl.__version__)
print("cuda:", torch.cuda.is_available())


torch: 2.1.0+cu121
dgl: 2.1.0+cu121
cuda: True


/home/yyyy/anaconda3/envs/gnn/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import dgl
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split

# ==========================
# 1. 图加载
# ==========================

def load_graph(node_csv, edge_csv, device="cpu"):
    nodes = pd.read_csv(node_csv)
    num_nodes = len(nodes)

    edges = pd.read_csv(edge_csv)
    src = torch.tensor(edges["src"].values, dtype=torch.int64)
    dst = torch.tensor(edges["dst"].values, dtype=torch.int64)

    g = dgl.graph((src, dst), num_nodes=num_nodes)
    g = dgl.add_edges(g, dst, src)
    g = dgl.add_self_loop(g)


    return g.to(device), edges, num_nodes


# ==========================
# 2. 构造训练样本
# ==========================
#TODO  这里采样的无边不一定是真的无边
def sample_non_edges(num_nodes, edges, num_samples):
    exist = set()
    for s, d in zip(edges["src"], edges["dst"]):
        exist.add((s, d))
        exist.add((d, s))  # 无向视角
    samples = set()
    print(len(exist))
    while len(samples) < num_samples:
        s = np.random.randint(0, num_nodes)
        d = np.random.randint(0, num_nodes)
        if s != d and (s, d) not in exist:
            samples.add((s, d))
    print(len(samples))
    return samples


def build_samples(g, edges, num_nodes):
    pos = edges[edges["label"] == 1]
    neg = edges[edges["label"] == 2]

    src, dst, label = [], [], []

    for _, r in pos.iterrows():
        src.append(r.src)
        dst.append(r.dst)
        label.append(1)

    for _, r in neg.iterrows():
        src.append(r.src)
        dst.append(r.dst)
        label.append(2)

    # 采样无边
    non_edges = sample_non_edges(num_nodes, edges, len(pos))
    for s, d in non_edges:
        src.append(s)
        dst.append(d)
        label.append(0)

    return torch.tensor(src), torch.tensor(dst), torch.tensor(label)


# ==========================
# Dataset
# ==========================

class EdgeDataset(Dataset):
    def __init__(self, src, dst, label):
        self.src = src
        self.dst = dst
        self.label = label

    def __len__(self):
        return len(self.label)

    def __getitem__(self, idx):
        return self.src[idx], self.dst[idx], self.label[idx]


# ==========================
# RGCN + 三分类 Decoder
# ==========================

from dgl.nn import GraphConv

class RGCNEncoder(nn.Module):
    def __init__(self, in_feats=16, h_feats=32, out_feats=32):
        super().__init__()
        self.layer1 = GraphConv(in_feats, h_feats)
        self.layer2 = GraphConv(h_feats, out_feats)

    def forward(self, g, feat):
        h = torch.relu(self.layer1(g, feat))
        h = self.layer2(g, h)
        return h

import torch.nn.functional as F
from dgl.nn.pytorch import GATConv


class GATEncoder(nn.Module):
    def __init__(self, in_dim=16, hidden_dim=32, out_dim=32, num_heads=4, dropout=0.2):
        super().__init__()
        self.dropout = dropout

        # 第一层：in_dim → hidden_dim
        self.layer1 = GATConv(
            in_dim,
            hidden_dim,
            num_heads=num_heads,
            feat_drop=0,
            attn_drop=0
        )

        # 第二层：hidden_dim → out_dim
        # 注意：上一层会输出 hidden_dim * num_heads
        self.layer2 = GATConv(
            hidden_dim,
            out_dim,
            num_heads=1,
            feat_drop=dropout,
            attn_drop=dropout
        )

    def forward(self, g, feat):
        # Layer1
        h = self.layer1(g, feat)       # shape: [N, num_heads, hidden_dim]
        h = h.mean(dim=1)              # 多头求平均 → [N, hidden_dim]
        h = F.elu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)

        # Layer2
        h = self.layer2(g, h)          # shape: [N, 1, out_dim]
        h = h.squeeze(1)               # → [N, out_dim]
        return h

class EdgeClassifier(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, num_classes=3):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, emb, src, dst):
        h = torch.cat([emb[src], emb[dst]], dim=1)
        return self.mlp(h)


class RGCNModel(nn.Module):
    def __init__(self, in_dim=16):
        super().__init__()
        self.encoder = RGCNEncoder(in_dim, 32, 32)
        self.decoder = EdgeClassifier(32, 64, 3)

    def forward(self, g, feat, src, dst):
        emb = self.encoder(g, feat)
        return self.decoder(emb, src, dst)

class GATModel(nn.Module):
    def __init__(self, in_dim=16, hidden_dim=32, out_dim=32, num_heads=4, dropout=0.2):
        super().__init__()
        self.encoder = GATEncoder(
            in_dim=in_dim,
            hidden_dim=hidden_dim,
            out_dim=out_dim,
            num_heads=num_heads,
            dropout=dropout
        )
        self.decoder = EdgeClassifier(out_dim)

    def forward(self, g, feat, src, dst):
        emb = self.encoder(g, feat)
        return self.decoder(emb, src, dst)
# ==========================
# Train & Eval
# ==========================
def multi_class_pr_auc(labels, probs, num_classes=3):
    prcs = []
    for c in range(num_classes):
        y_true = (labels == c).astype(int)
        y_score = probs[:, c]

        if y_true.sum() == 0:   # 跳过没有真实样本的类别
            continue

        prc = average_precision_score(y_true, y_score)
        prcs.append(prc)

    return np.mean(prcs) if len(prcs) > 0 else 0.0

# ==========================
# 1. 改进后的采样与数据准备
# ==========================

def sample_non_edges_fixed(num_nodes, forbidden_src, forbidden_dst, num_samples):
    """
    采样无边 (Label 0)。
    forbidden_src/dst: 必须包含所有真实存在的边 (Label 1 和 Label 2)，防止生成的 0 与 2 冲突。
    """
    exist = set()
    for s, d in zip(forbidden_src, forbidden_dst):
        if s > d: s, d = d, s # 统一存为 (min, max) 处理无向冲突
        exist.add((s, d))
    
    samples = set()
    # 简单的随机采样
    # 如果图非常稠密，这里建议加一个最大尝试次数防止死循环，但在一般稀疏图没问题
    while len(samples) < num_samples:
        s = np.random.randint(0, num_nodes)
        d = np.random.randint(0, num_nodes)
        
        if s == d: continue
        
        k = (s, d) if s < d else (d, s)
        if k not in exist:
            samples.add((s, d)) 
            
    return list(samples)

def prepare_data(node_csv, edge_csv):
    print("正在准备数据...")
    
    # 1. 读取基础数据
    nodes = pd.read_csv(node_csv)
    num_nodes = len(nodes)
    edges = pd.read_csv(edge_csv)
    
    # 2. 分离不同 Label 的边
    # 假设 CSV 中 label 列明确为 1 和 2
    df_pos = edges[edges['label'] == 1]  # Pos 边
    df_neg = edges[edges['label'] == 2]  # Neg 边 (真实存在的另一种关系)
    
    print(f"原始数据统计: Label 1 (Pos) = {len(df_pos)}, Label 2 (Neg) = {len(df_neg)}")

    # 3. 确定采样数量
    # 您的要求：Label 0 的数量 == Label 1 的数量
    num_samples_L0 = len(df_pos)
    
    # 4. 生成 Label 0 (无边)
    # 注意：forbidden 必须传入所有真实边 (Label 1 + Label 2)，防止把 Label 2 误当成 Label 0
    all_real_src = edges["src"].values
    all_real_dst = edges["dst"].values
    
    generated_L0_pairs = sample_non_edges_fixed(
        num_nodes, 
        forbidden_src=all_real_src, 
        forbidden_dst=all_real_dst, 
        num_samples=num_samples_L0
    )
    
    # 构造 Label 0 的数组
    src_L0 = np.array([p[0] for p in generated_L0_pairs])
    dst_L0 = np.array([p[1] for p in generated_L0_pairs])
    lbl_L0 = np.zeros(len(generated_L0_pairs), dtype=int)
    
    # 5. 合并所有数据 (Label 1 + Label 2 + Label 0)
    all_src = np.concatenate([edges["src"].values, src_L0])
    all_dst = np.concatenate([edges["dst"].values, dst_L0])
    all_lbl = np.concatenate([edges["label"].values, lbl_L0])
    
    print(f"采样后总数据量: {len(all_lbl)} (其中 Label 0: {len(lbl_L0)})")
    
    # 6. 划分训练/验证集
    all_idx = np.arange(len(all_lbl))
    train_idx, val_idx = train_test_split(all_idx, test_size=0.2, shuffle=True, random_state=42)
    
    # =====================================================
    # 7. 构建 GNN 背景图 (防泄露)
    # =====================================================
    # 规则：图里只能包含 "训练集" 里的 "真实边" (Label 1 和 Label 2)
    
    is_in_train = np.isin(all_idx, train_idx) 
    is_real_edge = (all_lbl != 0) # 只有 Label 1 和 2 能进图
    
    graph_edges_mask = is_in_train & is_real_edge
    
    g_src = all_src[graph_edges_mask]
    g_dst = all_dst[graph_edges_mask]
    
    # 构建 DGL 图
    g = dgl.graph((g_src, g_dst), num_nodes=num_nodes)
    g = dgl.add_edges(g, g_dst, g_src) # 添加反向边 (通常做法)
    g = dgl.add_self_loop(g)
    
    # 8. 封装 Dataset
    train_dataset = EdgeDataset(all_src[train_idx], all_dst[train_idx], all_lbl[train_idx])
    val_dataset = EdgeDataset(all_src[val_idx], all_dst[val_idx], all_lbl[val_idx])
    
    return g, train_dataset, val_dataset, num_nodes




def evaluate_unbalance(model, loader, g, feat, device):
    model.eval()
    all_labels = []
    all_preds = []
    all_probs = []

    criterion = nn.CrossEntropyLoss()
    total_loss = 0

    with torch.no_grad():
        for s, d, y in loader:
            s, d, y = s.to(device), d.to(device), y.to(device)

            logits = model(g, feat, s, d)
            loss = criterion(logits, y)
            total_loss += loss.item()

            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_labels.extend(y.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    labels = np.array(all_labels)
    preds = np.array(all_preds)
    probs = np.array(all_probs)

    # 基础指标
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    auc = roc_auc_score(labels, probs, multi_class="ovr")
    prc = multi_class_pr_auc(labels, probs, num_classes=3)
    pre = precision_score(labels, preds, average="macro", zero_division=0)
    rec = recall_score(labels, preds, average="macro", zero_division=0)
    # [关键修改] 获取每个类别的详细 Recall
    # labels=[0, 1, 2] 确保即使某个类没预测出来也不会报错
    rec_per_class = recall_score(labels, preds, average=None, labels=[0, 1, 2], zero_division=0)
    pre_per_class = precision_score(labels, preds, average=None, labels=[0, 1, 2], zero_division=0)
    
    # 提取 Label 2 (Negative) 的指标
    rec_label2 = rec_per_class[2]
    pre_label2 = pre_per_class[2]

    # 返回增加了 rec_label2
    return total_loss, acc, f1, auc, prc, pre,rec, rec_label2, pre_label2



def train_model_unbalance(node_csv, edge_csv):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # 1. 准备数据
    g, train_ds, val_ds, num_nodes = prepare_data(node_csv, edge_csv)
    g = g.to(device)

    # 2. Embedding
    node_emb = nn.Embedding(num_nodes, 16).to(device)
    
    # 3. DataLoader
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=256, shuffle=False)

    # 4. 模型 (RGCN 或 GAT)
    model = GATModel(in_dim=16, hidden_dim=32, out_dim=32, num_heads=4).to(device)
    # model = RGCNModel(in_dim=16).to(device)
    optimizer = optim.Adam(list(model.parameters()) + list(node_emb.parameters()), lr=1e-3)
    
    # [重要] 实验要求 Loss 不加权，以观察不平衡的影响
    criterion = nn.CrossEntropyLoss()

    # 记录最后一次的结果
    final_metrics = {}

    for epoch in range(20): # 20个epoch通常足够观察趋势
        model.train()
        total_loss = 0
        
        # 简化 tqdm 显示
        epoch_iter = tqdm(train_loader, desc=f"Epoch {epoch:02d}", ncols=80, leave=False)
        
        for s, d, y in epoch_iter:
            s, d, y = s.to(device), d.to(device), y.to(device)
            logits = model(g, node_emb.weight, s, d)
            loss = criterion(logits, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # 验证
        val_loss, acc, f1, auc, prc, pre, rec,rec2, pre2 = evaluate_unbalance(model, val_loader, g, node_emb.weight, device)

    # ====== 修改 1：打印所有指标 ======
    # 为了防止一行太长，这里使用了分行写法，保留了4位小数
    print(f"[Ep {epoch}] Val_Loss={val_loss:.4f} | Acc={acc:.4f} | F1={f1:.4f} | "
        f"AUC={auc:.4f} | AP={prc:.4f} | Neg_Rec={rec2:.4f} | Neg_Pre={pre2:.4f}")

    # ====== 修改 2：返回包含所有指标的字典 ======
    final_metrics = {
        "loss": val_loss,
        "acc": acc,
        "macro_f1": f1,
        "auc": auc,
        "ap": prc,             # Average Precision (PRC)
        "neg_recall": rec2,    # Label 2 (负样本) 召回率
        "neg_precision": pre2,  # Label 2 (负样本) 精确率
        "pre": pre,
        "rec": rec
    }
    return model, final_metrics

def evaluate(model, loader, g, feat, device):
    model.eval()
    all_labels = []
    all_preds = []
    all_probs = []

    criterion = nn.CrossEntropyLoss()
    total_loss = 0

    with torch.no_grad():
        for s, d, y in loader:
            s, d, y = s.to(device), d.to(device), y.to(device)

            logits = model(g, feat, s, d)
            loss = criterion(logits, y)
            total_loss += loss.item()

            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_labels.extend(y.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    labels = np.array(all_labels)
    preds = np.array(all_preds)
    probs = np.array(all_probs)

    acc = accuracy_score(labels, preds)
    pre = precision_score(labels, preds, average="macro", zero_division=0)
    rec = recall_score(labels, preds, average="macro", zero_division=0)
    f1 = f1_score(labels, preds, average="macro")

    # AUC（多分类用 OvR）
    auc = roc_auc_score(labels, probs, multi_class="ovr")
    prc = multi_class_pr_auc(labels, probs, num_classes=3)

    return total_loss, acc, pre, rec, f1, auc, prc


# ==========================
# 主函数
# ==========================

def train_model(node_csv, edge_csv):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # 1. 准备数据 (数量逻辑已修复)
    g, train_ds, val_ds, num_nodes = prepare_data(node_csv, edge_csv)
    g = g.to(device)

    # 2. 可学习的 Embedding
    node_emb = nn.Embedding(num_nodes, 16).to(device)
    
    # 3. DataLoader
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=256, shuffle=False)

    # 4. 模型
    model = GATModel(in_dim=16, hidden_dim=32, out_dim=32, num_heads=4).to(device)
    # model = RGCNModel(in_dim=16).to(device)
    optimizer = optim.Adam(list(model.parameters()) + list(node_emb.parameters()), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    # 5. Loop
    prc_list = []
    auc_list = []
    f1_list = []
    train_loss_list = []
    val_loss_list = []
    
    for epoch in range(20):
        model.train()
        total_loss = 0
        
        epoch_iter = tqdm(train_loader, desc=f"Epoch {epoch:02d}", ncols=100)
        
        for s, d, y in epoch_iter:
            s, d, y = s.to(device), d.to(device), y.to(device)
            
            logits = model(g, node_emb.weight, s, d)
            loss = criterion(logits, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            epoch_iter.set_postfix(loss=f"{loss.item():.4f}")

        # Eval
        val_loss, acc, pre, rec, f1, auc, prc = evaluate(model, val_loader, g, node_emb.weight, device)
        prc_list.append(prc)
        auc_list.append(auc)
        f1_list.append(f1)
        train_loss_list.append(total_loss)
        val_loss_list.append(val_loss)

        print(
            f"[Epoch {epoch:02d}] "
            f"Train Loss={total_loss:.4f} | "
            f"Val Loss={val_loss:.4f} | "
            f"Acc={acc:.4f} | Pre={pre:.4f} | Rec={rec:.4f} | "
            f"F1={f1:.4f} | AUC={auc:.4f} | PRC={prc:.4f}"
        )

    print("训练完成！")
    results = {
        "train_loss": train_loss_list,
        "val_loss": val_loss_list,
        "prc": prc_list,
        "auc": auc_list,
        "f1": f1_list
    }
    return model,results



/home/yyyy/anaconda3/envs/gnn/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import torch
import numpy as np
import random
import dgl
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def reset_seed(seed):
    np.random.seed(seed)  # 设置NumPy随机种子
    random.seed(seed)  # 设置Python随机种子

    torch.manual_seed(seed)
    dgl.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # 多GPU情况

    # torch.backends.cudnn.deterministic = True

seed = 42
reset_seed(seed)

# 正式测试

## Baseline

In [5]:
reset_seed(seed)
print("正在处理：", "/home/yyyy/codework/GARplus/experiments/exp1_accuracy/DDA_test/data_signed/baseline_edges.csv",)
model,cur_results = train_model("/home/yyyy/codework/GARplus/experiments/exp1_accuracy/DDA_test/data_signed/unified_node.csv", "/home/yyyy/codework/GARplus/experiments/exp1_accuracy/DDA_test/data_signed/baseline_edges.csv")
print(cur_results)
print("=============================================================================")

正在处理： /home/yyyy/codework/GARplus/experiments/exp1_accuracy/DDA_test/data_signed/baseline_edges.csv
正在准备数据...
原始数据统计: Label 1 (Pos) = 30956, Label 2 (Neg) = 4218
采样后总数据量: 66130 (其中 Label 0: 30956)


Epoch 00: 100%|█████████████████████████████████████| 827/827 [00:04<00:00, 172.77it/s, loss=0.4069]


[Epoch 00] Train Loss=419.2195 | Val Loss=28.1807 | Acc=0.8217 | Pre=0.5481 | Rec=0.5866 | F1=0.5667 | AUC=0.8416 | PRC=0.6140


Epoch 01: 100%|█████████████████████████████████████| 827/827 [00:05<00:00, 163.96it/s, loss=0.4430]


[Epoch 01] Train Loss=331.7756 | Val Loss=28.0522 | Acc=0.8239 | Pre=0.6605 | Rec=0.5885 | F1=0.5689 | AUC=0.8474 | PRC=0.6189


...

[Epoch 17] Train Loss=197.3085 | Val Loss=34.4600 | Acc=0.8276 | Pre=0.5925 | Rec=0.5980 | F1=0.5889 | AUC=0.8127 | PRC=0.6211


Epoch 18: 100%|█████████████████████████████████████| 827/827 [00:05<00:00, 163.42it/s, loss=0.1874]


[Epoch 18] Train Loss=194.5883 | Val Loss=34.0319 | Acc=0.8321 | Pre=0.5967 | Rec=0.6005 | F1=0.5906 | AUC=0.8140 | PRC=0.6224


Epoch 19: 100%|█████████████████████████████████████| 827/827 [00:04<00:00, 179.32it/s, loss=0.2089]


[Epoch 19] Train Loss=190.6167 | Val Loss=34.8221 | Acc=0.8272 | Pre=0.6017 | Rec=0.6020 | F1=0.5959 | AUC=0.8134 | PRC=0.6204
训练完成！
{'train_loss': [419.2195398956537, 331.77563901245594, 305.12490263581276, 289.30520172417164, 275.98905386030674, 266.88962161540985, 256.71383284032345, 250.17633777856827, 240.53737577050924, 234.0118130147457, 228.7477056235075, 223.07348568737507, 218.2649598941207, 214.13172002881765, 207.7166653573513, 205.68698943033814, 201.24614591896534, 197.3084782063961, 194.58829515054822, 190.61668191850185], 'val_loss': [28.180650293827057, 28.05217170715332, 28.36994281411171, 28.63926985859871, 29.14479449391365, 29.566014230251312, 30.45458275079727, 29.97347816824913, 30.979142755270004, 30.978862702846527, 30.787910640239716, 31.808313965797424, 31.52282103896141, 32.14286693930626, 32.483949303627014, 32.590725928545, 33.523557126522064, 34.45995658636093, 34.03186109662056, 34.82211884856224], 'prc': [0.6140207707451198, 0.6188709834425039, 0.616305

## LLM negatives

In [6]:
reset_seed(seed)
print("正在处理：", "/home/yyyy/codework/GARplus/experiments/exp1_accuracy/DDA_test/data_signed/llm_augmented_edges.csv",)
model,cur_results = train_model("/home/yyyy/codework/GARplus/experiments/exp1_accuracy/DDA_test/data_signed/unified_node.csv", "/home/yyyy/codework/GARplus/experiments/exp1_accuracy/DDA_test/data_signed/llm_augmented_edges.csv")
print(cur_results)
print("=============================================================================")

正在处理： /home/yyyy/codework/GARplus/experiments/exp1_accuracy/DDA_test/data_signed/llm_augmented_edges.csv
正在准备数据...
原始数据统计: Label 1 (Pos) = 30956, Label 2 (Neg) = 4218
采样后总数据量: 66130 (其中 Label 0: 30956)


Epoch 00: 100%|█████████████████████████████████████| 827/827 [00:04<00:00, 169.30it/s, loss=0.2671]


[Epoch 00] Train Loss=338.9553 | Val Loss=22.5531 | Acc=0.8537 | Pre=0.7996 | Rec=0.8061 | F1=0.8027 | AUC=0.9463 | PRC=0.8535


Epoch 01: 100%|█████████████████████████████████████| 827/827 [00:04<00:00, 178.39it/s, loss=0.2284]


[Epoch 01] Train Loss=219.5107 | Val Loss=21.0168 | Acc=0.8625 | Pre=0.8207 | Rec=0.8337 | F1=0.8268 | AUC=0.9539 | PRC=0.8878


...

[Epoch 17] Train Loss=113.7812 | Val Loss=22.5635 | Acc=0.8870 | Pre=0.8736 | Rec=0.8524 | F1=0.8618 | AUC=0.9706 | PRC=0.9286


Epoch 18: 100%|█████████████████████████████████████| 827/827 [00:06<00:00, 125.48it/s, loss=0.0610]


[Epoch 18] Train Loss=113.3834 | Val Loss=21.3364 | Acc=0.8867 | Pre=0.8847 | Rec=0.8270 | F1=0.8513 | AUC=0.9706 | PRC=0.9288


Epoch 19: 100%|█████████████████████████████████████| 827/827 [00:04<00:00, 174.25it/s, loss=0.1521]


[Epoch 19] Train Loss=110.9658 | Val Loss=22.0675 | Acc=0.8887 | Pre=0.8775 | Rec=0.8533 | F1=0.8643 | AUC=0.9710 | PRC=0.9291
训练完成！
{'train_loss': [338.9553012922406, 219.51070979237556, 193.5288860797882, 177.8632414266467, 165.98273561894894, 158.46440065652132, 150.17209762893617, 144.8416304886341, 139.12644588202238, 135.53556594252586, 131.85334071703255, 129.8516445532441, 126.82839642465115, 123.26803335919976, 121.57188014872372, 117.05983941257, 114.90190388634801, 113.78120760247111, 113.38337138481438, 110.9658114798367], 'val_loss': [22.55305388569832, 21.016839563846588, 21.24259017407894, 22.1496402323246, 21.17865453660488, 21.07642349600792, 21.308656707406044, 21.790004566311836, 20.687114343047142, 20.295456677675247, 21.369885683059692, 21.41302302479744, 20.96070359647274, 21.136948138475418, 21.090942785143852, 21.89953763782978, 22.056497156620026, 22.56345510482788, 21.336384773254395, 22.06754596531391], 'prc': [0.8534537169270707, 0.8877802787834451, 0.900667

## GAR Augment

In [7]:
reset_seed(seed)
print("正在处理：", "/home/yyyy/codework/GARplus/experiments/exp1_accuracy/DDA_test/data_signed/gar_augmented_edges.csv",)
model,cur_results = train_model("/home/yyyy/codework/GARplus/experiments/exp1_accuracy/DDA_test/data_signed/unified_node.csv", "/home/yyyy/codework/GARplus/experiments/exp1_accuracy/DDA_test/data_signed/gar_augmented_edges.csv")
print(cur_results)
print("=============================================================================")

正在处理： /home/yyyy/codework/GARplus/experiments/exp1_accuracy/DDA_test/data_signed/gar_augmented_edges.csv
正在准备数据...
原始数据统计: Label 1 (Pos) = 30956, Label 2 (Neg) = 4218
采样后总数据量: 66130 (其中 Label 0: 30956)


Epoch 00: 100%|█████████████████████████████████████| 827/827 [00:04<00:00, 175.21it/s, loss=0.2469]


[Epoch 00] Train Loss=313.0065 | Val Loss=19.6649 | Acc=0.8744 | Pre=0.8660 | Rec=0.8943 | F1=0.8791 | AUC=0.9587 | PRC=0.9411


Epoch 01: 100%|█████████████████████████████████████| 827/827 [00:04<00:00, 172.40it/s, loss=0.2262]


[Epoch 01] Train Loss=189.5878 | Val Loss=18.8161 | Acc=0.8784 | Pre=0.8891 | Rec=0.8999 | F1=0.8938 | AUC=0.9637 | PRC=0.9524


...

[Epoch 17] Train Loss=90.8821 | Val Loss=18.9439 | Acc=0.9050 | Pre=0.9219 | Rec=0.9304 | F1=0.9257 | AUC=0.9758 | PRC=0.9713


Epoch 18: 100%|█████████████████████████████████████| 827/827 [00:06<00:00, 128.22it/s, loss=0.2187]


[Epoch 18] Train Loss=90.2458 | Val Loss=17.9592 | Acc=0.9055 | Pre=0.9286 | Rec=0.9069 | F1=0.9173 | AUC=0.9760 | PRC=0.9719


Epoch 19: 100%|█████████████████████████████████████| 827/827 [00:04<00:00, 182.06it/s, loss=0.0505]


[Epoch 19] Train Loss=88.7872 | Val Loss=17.7202 | Acc=0.9096 | Pre=0.9268 | Rec=0.9322 | F1=0.9293 | AUC=0.9768 | PRC=0.9722
训练完成！
{'train_loss': [313.0064690336585, 189.5877951607108, 169.7648603208363, 153.8603134714067, 144.7611279860139, 136.5839643739164, 128.21613791957498, 124.41051492467523, 120.56893718242645, 112.56441610306501, 110.52517333254218, 105.851578341797, 104.10934981703758, 102.2142643975094, 99.35156050883234, 95.67635215632617, 93.37953660823405, 90.88209097925574, 90.24582720920444, 88.78723220806569], 'val_loss': [19.664910182356834, 18.816130235791206, 17.506650909781456, 16.794552370905876, 16.053815945982933, 16.27918180823326, 16.344836562871933, 16.04933652281761, 16.36349993944168, 16.62066523730755, 16.956901028752327, 17.60652920603752, 16.900552302598953, 16.67151254415512, 17.431896537542343, 16.97331230342388, 17.65536916255951, 18.94385699927807, 17.959199741482735, 17.72016640007496], 'prc': [0.9411354759024326, 0.9523550974003635, 0.960432364023

# Added negatives

In [5]:
reset_seed(seed)
print("正在处理：", "/home/yyyy/codework/GARplus/GNN/code/data_updated/edge_update.csv",)
model,cur_results = train_model("/home/yyyy/codework/GARplus/GNN/code/data_updated/node.csv", "/home/yyyy/codework/GARplus/GNN/code/data_updated/edge_update.csv")
print(cur_results)
print("=============================================================================")

正在处理： /home/yyyy/codework/GARplus/GNN/code/data_updated/edge_update.csv
正在准备数据...
原始数据统计: Label 1 (Pos) = 685559, Label 2 (Neg) = 10983
采样后总数据量: 1382101 (其中 Label 0: 685559)


Epoch 00: 100%|█████████████████████████████████| 17277/17277 [02:14<00:00, 128.49it/s, loss=0.2959]


[Epoch 00] Train Loss=6734.3460 | Val Loss=462.4027 | Acc=0.8142 | Pre=0.8030 | Rec=0.6498 | F1=0.6887 | AUC=0.9303 | PRC=0.7671


Epoch 01: 100%|█████████████████████████████████| 17277/17277 [02:11<00:00, 130.97it/s, loss=0.1805]


[Epoch 01] Train Loss=5500.4781 | Val Loss=373.3324 | Acc=0.8578 | Pre=0.7724 | Rec=0.7784 | F1=0.7744 | AUC=0.9479 | PRC=0.8304


...

[Epoch 17] Train Loss=4175.7340 | Val Loss=321.7061 | Acc=0.8821 | Pre=0.8421 | Rec=0.7754 | F1=0.8033 | AUC=0.9629 | PRC=0.8733


Epoch 18: 100%|█████████████████████████████████| 17277/17277 [02:14<00:00, 128.12it/s, loss=0.3470]


[Epoch 18] Train Loss=4162.9375 | Val Loss=319.6970 | Acc=0.8830 | Pre=0.8311 | Rec=0.8079 | F1=0.8187 | AUC=0.9633 | PRC=0.8794


Epoch 19: 100%|█████████████████████████████████| 17277/17277 [02:16<00:00, 126.33it/s, loss=0.2432]


[Epoch 19] Train Loss=4141.8360 | Val Loss=324.1527 | Acc=0.8809 | Pre=0.8602 | Rec=0.7664 | F1=0.8031 | AUC=0.9626 | PRC=0.8777
训练完成！
{'train_loss': [6734.346001803875, 5500.478099107742, 5149.372123070061, 4957.209173135459, 4820.667921371758, 4705.434025719762, 4629.121860861778, 4557.647487446666, 4501.588297296315, 4441.3027918078005, 4395.254830002785, 4365.333824895322, 4323.099011767656, 4290.432022500783, 4266.724802225828, 4233.697186481208, 4209.9744760766625, 4175.733950033784, 4162.937485106289, 4141.83599871397], 'val_loss': [462.4026908874512, 373.33239959180355, 381.2492921203375, 372.23230643570423, 344.51278878748417, 339.733305811882, 337.49300932884216, 334.23559752106667, 333.8100264817476, 333.4828023612499, 326.2794011682272, 330.14583918452263, 347.10915207862854, 336.23628617823124, 324.82026466727257, 327.8180863112211, 322.7172889113426, 321.7060983777046, 319.6969923377037, 324.1526627987623], 'prc': [0.7670547654196697, 0.8304492348241347, 0.845829830670553

# Original negatives

In [6]:
reset_seed(seed)
print("正在处理：", "/home/yyyy/codework/GARplus/GNN/code/data_updated/edge_old.csv",)
model,cur_results = train_model("/home/yyyy/codework/GARplus/GNN/code/data_updated/node.csv", "/home/yyyy/codework/GARplus/GNN/code/data_updated/edge_old.csv")
print(cur_results)
print("=============================================================================")

正在处理： /home/yyyy/codework/GARplus/GNN/code/data_updated/edge_old.csv
正在准备数据...
原始数据统计: Label 1 (Pos) = 685559, Label 2 (Neg) = 6861
采样后总数据量: 1377979 (其中 Label 0: 685559)


Epoch 00: 100%|█████████████████████████████████| 17225/17225 [02:15<00:00, 127.58it/s, loss=0.3708]


[Epoch 00] Train Loss=6577.3320 | Val Loss=416.8250 | Acc=0.8355 | Pre=0.7281 | Rec=0.5601 | F1=0.5584 | AUC=0.9092 | PRC=0.6483


Epoch 01: 100%|█████████████████████████████████| 17225/17225 [02:14<00:00, 127.94it/s, loss=0.1935]


[Epoch 01] Train Loss=5483.7647 | Val Loss=397.7396 | Acc=0.8447 | Pre=0.7391 | Rec=0.6333 | F1=0.6608 | AUC=0.9398 | PRC=0.7291


...

[Epoch 17] Train Loss=4167.1684 | Val Loss=323.2045 | Acc=0.8830 | Pre=0.7845 | Rec=0.7258 | F1=0.7497 | AUC=0.9611 | PRC=0.7986


Epoch 18: 100%|█████████████████████████████████| 17225/17225 [02:09<00:00, 132.64it/s, loss=0.2076]


[Epoch 18] Train Loss=4138.5517 | Val Loss=323.5338 | Acc=0.8831 | Pre=0.8090 | Rec=0.7044 | F1=0.7399 | AUC=0.9611 | PRC=0.8098


Epoch 19: 100%|█████████████████████████████████| 17225/17225 [02:12<00:00, 130.14it/s, loss=0.1591]


[Epoch 19] Train Loss=4119.4504 | Val Loss=326.3133 | Acc=0.8831 | Pre=0.8026 | Rec=0.7098 | F1=0.7429 | AUC=0.9608 | PRC=0.8026
训练完成！
{'train_loss': [6577.331985831261, 5483.764661565423, 5097.22921204567, 4899.193782605231, 4767.317020997405, 4665.528185259551, 4575.965968005359, 4515.136567927897, 4447.836058229208, 4401.815518438816, 4369.719866618514, 4325.326356954873, 4303.343732506037, 4265.323348540813, 4236.061434622854, 4211.624702118337, 4191.915673352778, 4167.168373942375, 4138.551675196737, 4119.450385224074], 'val_loss': [416.8250351846218, 397.73958753049374, 379.465218514204, 355.751649916172, 355.86449670791626, 339.3583353757858, 370.8822079002857, 331.6733053922653, 330.7238690108061, 335.61715792119503, 328.5997978001833, 327.3672527372837, 327.29526099562645, 323.1344450265169, 332.00676679611206, 326.5126682817936, 323.8484788388014, 323.2045142799616, 323.53377786278725, 326.31333215534687], 'prc': [0.6482820245890841, 0.7291132502810628, 0.7645825297528793, 0.

# Random negatives

In [7]:
reset_seed(seed)
print("正在处理：", "/home/yyyy/codework/GARplus/GNN/code/data_updated/edge_random.csv",)
model,cur_results = train_model("/home/yyyy/codework/GARplus/GNN/code/data_updated/node.csv", "/home/yyyy/codework/GARplus/GNN/code/data_updated/edge_random.csv")
print(cur_results)
print("=============================================================================")

正在处理： /home/yyyy/codework/GARplus/GNN/code/data_updated/edge_random.csv
正在准备数据...
原始数据统计: Label 1 (Pos) = 685559, Label 2 (Neg) = 6861
采样后总数据量: 1377979 (其中 Label 0: 685559)


Epoch 00: 100%|█████████████████████████████████| 17225/17225 [02:12<00:00, 129.94it/s, loss=0.3082]


[Epoch 00] Train Loss=6620.6774 | Val Loss=436.8648 | Acc=0.8285 | Pre=0.5590 | Rec=0.5554 | F1=0.5531 | AUC=0.8398 | PRC=0.6142


Epoch 01: 100%|█████████████████████████████████| 17225/17225 [02:08<00:00, 133.96it/s, loss=0.2209]


[Epoch 01] Train Loss=5670.7747 | Val Loss=410.4319 | Acc=0.8419 | Pre=0.5675 | Rec=0.5643 | F1=0.5622 | AUC=0.8503 | PRC=0.6228


...

[Epoch 17] Train Loss=4373.3189 | Val Loss=337.0452 | Acc=0.8828 | Pre=0.5887 | Rec=0.5917 | F1=0.5901 | AUC=0.8744 | PRC=0.6349


Epoch 18: 100%|█████████████████████████████████| 17225/17225 [02:08<00:00, 134.43it/s, loss=0.1393]


[Epoch 18] Train Loss=4348.8459 | Val Loss=338.0438 | Acc=0.8820 | Pre=0.5881 | Rec=0.5911 | F1=0.5895 | AUC=0.8714 | PRC=0.6344


Epoch 19: 100%|█████████████████████████████████| 17225/17225 [02:12<00:00, 130.12it/s, loss=0.2988]


[Epoch 19] Train Loss=4331.3015 | Val Loss=340.8990 | Acc=0.8829 | Pre=0.5887 | Rec=0.5917 | F1=0.5902 | AUC=0.8746 | PRC=0.6349
训练完成！
{'train_loss': [6620.677390038967, 5670.774680607021, 5358.563505619764, 5178.27936668694, 5054.88303745538, 4948.759101971984, 4877.7565547302365, 4808.458558615297, 4729.374538682401, 4671.167739223689, 4618.87580794096, 4565.839981473982, 4525.770088326186, 4486.627419047058, 4453.106632255018, 4429.749999009073, 4399.599142514169, 4373.318881094456, 4348.845931816846, 4331.301487471908], 'val_loss': [436.86484944820404, 410.4319260418415, 389.0546129196882, 389.6257016658783, 369.2459894865751, 361.9252206236124, 393.33110082149506, 347.65902762115, 346.34529519081116, 351.00120155513287, 343.6315079331398, 342.2324706017971, 342.5553871691227, 340.4439424276352, 344.2322402447462, 339.1030054539442, 338.6237618923187, 337.04522570967674, 338.0438126921654, 340.89895655214787], 'prc': [0.6142354570218671, 0.6227576052969529, 0.6270209135638375, 0.62

# new random test

In [10]:
reset_seed(seed)
print("正在处理：", "/home/yyyy/codework/GARplus/GNN/code/data_updated/edge_random_new.csv",)
model,cur_results = train_model("/home/yyyy/codework/GARplus/GNN/code/data_updated/node.csv", "/home/yyyy/codework/GARplus/GNN/code/data_updated/edge_random_new.csv")
print(cur_results)
print("=============================================================================")

正在处理： /home/yyyy/codework/GARplus/GNN/code/data_updated/edge_random_new.csv
正在准备数据...
原始数据统计: Label 1 (Pos) = 685559, Label 2 (Neg) = 10983
采样后总数据量: 1382101 (其中 Label 0: 685559)


Epoch 00: 100%|█████████████████████████████████| 17277/17277 [02:14<00:00, 128.47it/s, loss=0.3115]


[Epoch 00] Train Loss=6839.4550 | Val Loss=483.8746 | Acc=0.8107 | Pre=0.5555 | Rec=0.5449 | F1=0.5407 | AUC=0.8575 | PRC=0.6275


Epoch 01: 100%|█████████████████████████████████| 17277/17277 [02:17<00:00, 126.03it/s, loss=0.2339]


[Epoch 01] Train Loss=5793.5952 | Val Loss=433.3342 | Acc=0.8346 | Pre=0.7789 | Rec=0.5879 | F1=0.6060 | AUC=0.8835 | PRC=0.6795


...

[Epoch 17] Train Loss=4461.8596 | Val Loss=349.2235 | Acc=0.8758 | Pre=0.7839 | Rec=0.6757 | F1=0.7080 | AUC=0.9037 | PRC=0.7364


Epoch 18: 100%|█████████████████████████████████| 17277/17277 [02:14<00:00, 128.17it/s, loss=0.1832]


[Epoch 18] Train Loss=4442.0522 | Val Loss=341.2381 | Acc=0.8817 | Pre=0.7901 | Rec=0.6799 | F1=0.7130 | AUC=0.8999 | PRC=0.7369


Epoch 19: 100%|█████████████████████████████████| 17277/17277 [02:16<00:00, 126.94it/s, loss=0.4368]


[Epoch 19] Train Loss=4432.3467 | Val Loss=339.7572 | Acc=0.8820 | Pre=0.8349 | Rec=0.6399 | F1=0.6703 | AUC=0.9003 | PRC=0.7349
训练完成！
{'train_loss': [6839.455024898052, 5793.595202952623, 5451.020360149443, 5254.428123563528, 5114.272446028888, 5011.9321773946285, 4919.199236616492, 4849.868160478771, 4785.092904962599, 4739.890008356422, 4682.800240971148, 4646.445713169873, 4611.503565073013, 4582.3513714410365, 4548.848610699177, 4526.798485174775, 4491.939326245338, 4461.859622847289, 4442.052225008607, 4432.346687674522], 'val_loss': [483.87459766864777, 433.3342380821705, 394.80888080596924, 380.5418277978897, 363.8287379145622, 358.29847444593906, 360.6075882166624, 353.35752116143703, 350.5479025989771, 347.3708605468273, 348.72699804604053, 347.6504339873791, 348.6414155513048, 362.6890530437231, 353.17802433669567, 348.5192674845457, 343.7563985437155, 349.2234850227833, 341.23806308209896, 339.75719782710075], 'prc': [0.6274813445997726, 0.6795102608929056, 0.69812166375573

In [1]:
import pandas as pd

# baseline
original = {
    "Train Loss": 4119.4504,
    "Val Loss": 326.3133,
    "Acc": 0.8831,
    "Pre": 0.8026,
    "Rec": 0.7098,
    "F1": 0.7429,
    "AUC": 0.9608,
    "PRC": 0.8026
}

# updated group 1
update_GAR = {
    "Train Loss": 4141.8360,
    "Val Loss": 324.1527,
    "Acc": 0.8809,
    "Pre": 0.8602,
    "Rec": 0.7664,
    "F1": 0.8031,
    "AUC": 0.9626,
    "PRC": 0.8777
}

# updated group 2
random_updated = {
    "Train Loss": 4432.3467,
    "Val Loss": 339.7572,
    "Acc": 0.8820,
    "Pre": 0.8349,
    "Rec": 0.6399,
    "F1": 0.6703,
    "AUC": 0.9003,
    "PRC": 0.7349
}

# 构建 DataFrame
df = pd.DataFrame({
    "Original": original,
    "update_GAR": update_GAR,
    "random_updated": random_updated
})

# 计算变化百分比（相对 original）
df["Δ update_GAR (%)"] = (df["update_GAR"] - df["Original"]) / df["Original"] * 100
df["Δ random_updated (%)"] = (df["random_updated"] - df["Original"]) / df["Original"] * 100

# 保留小数
df = df.round(4)
df[["Δ update_GAR (%)", "Δ random_updated (%)"]] = df[
    ["Δ update_GAR (%)", "Δ random_updated (%)"]
].round(2)

print(df)


             Original  update_GAR  random_updated  Δ update_GAR (%)  \
Train Loss  4119.4504   4141.8360       4432.3467              0.54   
Val Loss     326.3133    324.1527        339.7572             -0.66   
Acc            0.8831      0.8809          0.8820             -0.25   
Pre            0.8026      0.8602          0.8349              7.18   
Rec            0.7098      0.7664          0.6399              7.97   
F1             0.7429      0.8031          0.6703              8.10   
AUC            0.9608      0.9626          0.9003              0.19   
PRC            0.8026      0.8777          0.7349              9.36   

            Δ random_updated (%)  
Train Loss                  7.60  
Val Loss                    4.12  
Acc                        -0.12  
Pre                         4.02  
Rec                        -9.85  
F1                         -9.77  
AUC                        -6.30  
PRC                        -8.44  
